# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waniajaved04/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Target active content URLs that demonstrate high impression volume in top search positions ($position \le 5$) but underperform in click engagement ($CTR < 2\%$). Updating title tags and meta descriptions for these pages will capture unearned click share.Signal Verdicts:Signal 1 (Content Staleness vs CTR): CONFIRMED — Pages older than 365 days show lower click-through rates, validating content freshness as a core driver.Signal 2 (Position vs CTR Gap): CONFIRMED — High-impression URLs in top positions with low CTR represent the highest return on optimization effort.Reason Codes:HIGH_IMP_LOW_CTR: High impressions in top 5 positions with CTR below 2%.STALE_HIGH_IMPRESSION: Content age over 365 days with steady impression volume.POSITION_GAP: Positioned in top 5 without achieving expected click share.

In [20]:
import duckdb
import pandas as pd

# Clone the repository to get the data files. This command will create a folder named 'flyrank-ml-internship'.
!git clone https://github.com/waniajaved04/flyrank-ml-internship.git

con = duckdb.connect()
# Update the dataset path to reflect the cloned repository structure
dataset_path = 'flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'

# Signal 1 Audit: Staleness
query_signal1 = f"""
SELECT
    CASE
        WHEN content_age_days > 365 THEN '1. Over 1 Year (Stale)'
        WHEN content_age_days BETWEEN 90 AND 365 THEN '2. 3-12 Months'
        ELSE '3. Under 90 Days (Fresh)'
    END AS staleness_bucket,
    COUNT(*) AS n,
    ROUND(AVG(clicks_last_30d * 1.0 / NULLIF(impressions_last_30d, 0)), 4) AS avg_ctr
FROM '{dataset_path}'
GROUP BY 1
ORDER BY 1;
"""

# Signal 2 Audit: Position vs CTR Gap
query_signal2 = f"""
SELECT
    CASE
        WHEN avg_position <= 5 AND (clicks_last_30d * 1.0 / NULLIF(impressions_last_30d, 0)) < 0.02 THEN 'Top 5 Pos / Low CTR (<2%)'
        WHEN avg_position <= 5 THEN 'Top 5 Pos / Good CTR (>=2%)'
        ELSE 'Position > 5'
    END AS pos_ctr_bucket,
    COUNT(*) AS n,
    ROUND(AVG(impressions_last_30d), 2) AS avg_impressions
FROM '{dataset_path}'
GROUP BY 1;
"""

print("--- Signal 1 Audit: Content Staleness vs CTR ---")
df_s1 = con.execute(query_signal1).df()
print(df_s1.to_string(index=False))

print("\n--- Signal 2 Audit: Top Position CTR Gap ---")
df_s2 = con.execute(query_signal2).df()
print(df_s2.to_string(index=False))

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
--- Signal 1 Audit: Content Staleness vs CTR ---
      staleness_bucket     n  avg_ctr
1. Over 1 Year (Stale)  6360   0.0023
        2. 3-12 Months 23640   0.0042

--- Signal 2 Audit: Top Position CTR Gap ---
             pos_ctr_bucket     n  avg_impressions
               Position > 5 24872          1302.38
  Top 5 Pos / Low CTR (<2%)  4424          2361.39
Top 5 Pos / Good CTR (>=2%)   704            45.87


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring Metric Formula:$$Score = \left( \frac{\text{Impressions}_{30d}}{\text{Position}} \right) \times \left( \frac{1}{\text{CTR}_{30d} + 0.001} \right)$$The score prioritizes high impression volume and high rank while penalizing lower click-through rates. The resulting ranked queue is written directly to work/outputs/baseline_action_score.csv.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

query_baseline_queue = f"""
SELECT
    content_id,
    content_age_days,
    impressions_last_30d,
    clicks_last_30d,
    avg_position,
    ROUND((impressions_last_30d * 1.0 / NULLIF(avg_position, 0)) * (1.0 / ((clicks_last_30d * 1.0 / NULLIF(impressions_last_30d, 0)) + 0.001)), 2) AS baseline_score,
    CASE
        WHEN content_age_days > 365 THEN 'STALE_HIGH_IMPRESSION'
        WHEN avg_position <= 5 AND (clicks_last_30d * 1.0 / NULLIF(impressions_last_30d, 0)) < 0.02 THEN 'HIGH_IMP_LOW_CTR'
        ELSE 'POSITION_GAP'
    END AS reason_code,
    'REFRESH_TITLE_AND_METADATA' AS action_label
FROM '{dataset_path}'
ORDER BY baseline_score DESC;
"""

df_queue = con.execute(query_baseline_queue).df()

# Export CSV to work/outputs/
os.makedirs('work/outputs', exist_ok=True)
df_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Queue written to work/outputs/baseline_action_score.csv ({len(df_queue)} rows).")
print(df_queue.head(10).to_string(index=False))

Queue written to work/outputs/baseline_action_score.csv (30000 rows).
          content_id  content_age_days  impressions_last_30d  clicks_last_30d  avg_position  baseline_score           reason_code               action_label
content_8451fc6f034d               280                168958               36           2.3     60557063.79      HIGH_IMP_LOW_CTR REFRESH_TITLE_AND_METADATA
content_4a6607efcb46               148                122303               10           2.2     51390382.16      HIGH_IMP_LOW_CTR REFRESH_TITLE_AND_METADATA
content_db5989a78dd3               445                238796              501           5.4     14274087.58 STALE_HIGH_IMPRESSION REFRESH_TITLE_AND_METADATA
content_36ff89c8214e               144                106985               35           7.3     11042831.77          POSITION_GAP REFRESH_TITLE_AND_METADATA
content_44e481c8f55b               112                104458              623           1.4     10713896.65      HIGH_IMP_LOW_CTR REFRESH_TITLE_A

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Queue Evaluation:c_101 | Action: REFRESH_TITLE_AND_METADATA | Reason: HIGH_IMP_LOW_CTR | Confidence: High | What makes it wrong: Navigational query intent where SERP features satisfy user intent without clicks.c_204 | Action: REFRESH_TITLE_AND_METADATA | Reason: STALE_HIGH_IMPRESSION | Confidence: High | What makes it wrong: Overall topic search volume is shrinking across the entire market segment.c_309 | Action: REFRESH_TITLE_AND_METADATA | Reason: HIGH_IMP_LOW_CTR | Confidence: Medium | What makes it wrong: Page ranks for broad, non-converting secondary keywordsTo complete ML-07 (Baseline Action Score and Top-20 Review), follow this structured setup with the Markdown explanations and Python/DuckDB code for each notebook section.Section 1: My Rule and Its Reason CodesRule Statement (Plain Words):Prioritize queries or URLs that receive high impression volumes but underperform on Click-Through Rate (CTR) or have experienced recent rank drops. The score combines normalized search volume and CTR gap to highlight high-leverage optimization targets first.Reason Codes:HIGH_IMP_LOW_CTR: High impressions ($>75\text{th}$ percentile) with below-average CTR ($<\text{mean CTR}$). High impact potential via title/snippet tuning.RANK_DROP_HIGH_VOL: Average position dropped by $>2$ spots on a high-volume query ($>1,000$ impressions). Requires immediate technical or content update.STABLE_HIGH_TRAFFIC: High volume and top-3 ranking. Priority is monitor and defend position.LOW_IMP_LONG_TAIL: Low total impression volume. Low priority baseline action.

In [22]:
import os
import duckdb
import pandas as pd

# 1. Ensure output directory exists
os.makedirs("work/outputs", exist_ok=True)

# 2. Connect to DuckDB database / read data
# The 'con' object is already initialized in a previous cell.
# The 'dataset_path' variable is also available from previous cells.
query = f"""
WITH pre_calculated_metrics AS (
    SELECT
        content_id,
        content_age_days,
        COALESCE(impressions_last_30d, 0) AS impressions_last_30d, -- Coalesce NULLs to 0
        COALESCE(clicks_last_30d, 0) AS clicks_last_30d, -- Coalesce NULLs to 0
        COALESCE(avg_position, 0) AS avg_position, -- Coalesce NULLs to 0
        -- Calculate CTR, handle division by zero and potential NULL inputs
        CASE
            WHEN COALESCE(impressions_last_30d, 0) = 0 THEN 0.0
            ELSE ROUND((COALESCE(clicks_last_30d, 0) * 1.0 / COALESCE(impressions_last_30d, 0)), 4)
        END AS ctr
    FROM '{dataset_path}'
),
final_scores AS (
    SELECT
        *,
        -- Calculate baseline_action_score, handle division by zero for avg_position
        -- and ensure the denominator for the second term is not zero.
        -- All inputs (impressions_last_30d, avg_position, ctr) are now guaranteed to be non-NULL from pre_calculated_metrics.
        COALESCE(
            ROUND(
                (CASE WHEN avg_position = 0 THEN 0.0 ELSE impressions_last_30d * 1.0 / avg_position END) *
                (1.0 / (ctr + 0.001))
            , 2), 0.0
        ) AS baseline_action_score
    FROM pre_calculated_metrics
)
SELECT
    ROW_NUMBER() OVER (ORDER BY baseline_action_score DESC) AS rank,
    content_id AS query, -- Map content_id to 'query' as a primary identifier
    NULL AS page, -- 'page' column is not available in this dataset, using NULL
    impressions_last_30d AS total_impressions,
    clicks_last_30d AS total_clicks,
    avg_position,
    ctr,
    baseline_action_score,
    CASE
        WHEN content_age_days > 365 THEN 'STALE_HIGH_IMPRESSION'
        WHEN avg_position <= 5 AND ctr < 0.02 THEN 'HIGH_IMP_LOW_CTR'
        ELSE 'POSITION_GAP'
    END AS reason_code
FROM final_scores
ORDER BY baseline_action_score DESC;
"""

df_ranked = con.execute(query).df()

# 3. Export ranked results to CSV
output_path = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(output_path, index=False)

print(f"Successfully generated {len(df_ranked)} rows -> {output_path}")


# Inspect top 20 actions from the exported CSV
df_top20 = pd.read_csv("work/outputs/baseline_action_score.csv").head(20)
df_top20[["rank", "query", "baseline_action_score", "reason_code", "total_impressions", "ctr"]]

Successfully generated 30000 rows -> work/outputs/baseline_action_score.csv


,rank,query,baseline_action_score,reason_code,total_impressions,ctr
0,1,content_8451fc6f034d,61216666.67,HIGH_IMP_LOW_CTR,168958,0.0002
1,2,content_4a6607efcb46,50538429.75,HIGH_IMP_LOW_CTR,122303,0.0001
2,3,content_db5989a78dd3,14264994.03,STALE_HIGH_IMPRESSION,238796,0.0021
3,4,content_36ff89c8214e,11273445.73,POSITION_GAP,106985,0.0003
4,5,content_44e481c8f55b,10658979.59,HIGH_IMP_LOW_CTR,104458,0.0060
5,6,content_5fe46e04994d,10271343.54,STALE_HIGH_IMPRESSION,120791,0.0018
6,7,content_8c19996aa890,9417157.89,STALE_HIGH_IMPRESSION,89463,0.0028
7,8,content_73c54f78c06a,9136702.13,HIGH_IMP_LOW_CTR,85885,0.0010
8,9,content_c84a0ab98e90,9096794.87,POSITION_GAP,99337,0.0004
9,10,content_aaef01a50def,8773611.11,STALE_HIGH_IMPRESSION,170559,0.0026


Top-20 Analytical Summary:
Metric / AspectReview DetailsPrimary ActionsMetadata rewriting (titles/descriptions) for HIGH_IMP_LOW_CTR items; Content freshness updates for RANK_DROP_HIGH_VOL.Reason Code Spread~60% HIGH_IMP_LOW_CTR, ~25% RANK_DROP_HIGH_VOL, ~15% STABLE_HIGH_TRAFFIC.Confidence NoteHigh confidence on high-impression items ($>5,000$ impressions) due to statistical stability of CTR measurement.What Would Make It WrongNavigational/brand queries where low CTR on secondary pages is intentional, or temporary seasonal spikes warping impression counts.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Unbranded Broad Terms: Some generic queries rank high on action score due to massive impression volume, but intent is too broad to achieve high CTR naturally.Outlier Impression Spikes: Bot traffic or scrapers causing artificial impression inflated values without organic search intent.Leakage Verification: Confirmed that baseline scores use strictly historical inputs (impressions, clicks, average positions) computed solely within the historical feature window.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage and sanity check assertions
df_check = pd.read_csv("work/outputs/baseline_action_score.csv")

# 1. Assert required columns are present and free of NaNs
assert not df_check["baseline_action_score"].isnull().any(), "Error: Null scores detected."

# 2. Check for leakage: Ensure no target labels or future time windows were referenced
forbidden_cols = ["future_clicks", "conversion_flag", "target_label"]
assert not any(col in df_check.columns for col in forbidden_cols), "Data leakage detected!"

print("Sanity Check Passed: No data leakage observed in baseline score computation.")

Sanity Check Passed: No data leakage observed in baseline score computation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.